# 🎯 [ISR02_M02] 게시물 텍스트 불용어 정제, 이미지 해시(pHash) 추출 및 중복 제거 전처리 모듈 구현
### Google Colab 실습 환경 (정보·감시·정찰(ISR) / Level 2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/ISR02/isr02_m02_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 정보·감시·정찰(ISR) (Level 2) |
| **미션 ID** | ISR02_M02 |
| **미션 주제** | 게시물 텍스트 불용어 정제, 이미지 해시(pHash) 추출 및 중복 제거 전처리 모듈 구현 |
| **소요 시간** | 약 2시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
러시아-우크라이나 전쟁에서는 텔레그램·X 같은 SNS에 현장 제보와 함께 재게시, 수치·지역 변조, 과거 사진 재활용, 생성형 AI 조작물, 조직적 동시 유포가 한꺼번에 쏟아지면서 공개출처정보(OSINT)의 출처와 신뢰도를 빠르게 가려내는 일이 정보전의 핵심이 되었다. 본 프로젝트에서는 가상 계정과 가상 지역 코드로 만든 교육용 합성 게시물을 사용해 원천 데이터 스키마 정의, 텍스트 정제와 이미지 지각 해시 기반 중복 제거, 유사도 기반 복제·1차 출처 분류 Baseline, 조작·기만 탐지와 계정 신뢰도 결합 모델, 종합 신뢰도 등급(A~E) 엔진, 정보전 모니터링 대시보드, 정보관 긴급 검토(HITL), 대량 스트림 부하 시험, AAR 재보정, 최종 통합까지 단계별로 구현한다. 최종 목표는 조작 게시물을 놓치지 않으면서 사실 제보를 가짜로 몰지 않고, 파급력이 큰 첩보의 채택·차단은 원천 출처를 확인한 정보관이 결정하도록 설계하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📂 2. 산출물 디렉토리 체계 및 실습 가이드

본 과제의 미션별 산출물(`deliverables`)은 단순 스크립트를 넘어 실무 엔터프라이즈 및 국방 지휘통제체계에 즉시 배포할 수 있는 **표준 패키지 디렉토리 구조**를 따릅니다:

| 디렉토리 | 주요 산출물(Deliverables) | 역할 및 설명 |
| :--- | :--- | :--- |
| **`data/`** | `data/schema/isr02_input_schema.json` | 입출력 데이터 표준 스키마 및 원천/정제 데이터셋 |
| **`src/`** | `src/preprocess/`, `src/models/`, `src/verification/`, `src/ui/`, `src/hitl/` | 전처리, Baseline, 앙상블 탐지, 대시보드, HITL 등 핵심 파이썬 소스 코드 모듈 |
| **`docs/`** | `docs/isr02_data_spec.md`, `docs/isr02_hitl_guide.md`, `docs/isr02_aar_report.md`, `docs/isr02_final_report.md` | 데이터 규격서, HITL 가이드, AAR 분석 보고서, 최종 브리핑 등 군사 행정/지휘관 문서 |
| **`configs/`**| `configs/isr02_tuned_params.yaml` | AAR 피드백 기반 재조정 임계치 및 환경설정 파일 |
| **`tests/`** | `tests/isr02_stress_test.py` | 500% 부하 시험 및 단위/통합 테스트 스위트 |

> **💡 [Google Colab / 주피터 노트북 실습 팁]**
> - **방법 A. 자동 파일 저장 (권장):** 코드 셀 첫 줄에 `%%writefile src/preprocess/isr02_preprocessor.py`와 같이 매직 명령어를 추가하여 실행하면, 노트북에서 작업한 코드가 해당 폴더의 파일로 자동 추출·저장됩니다.
> - **방법 B. 논리적 모듈 대응:** 별도 파일 추출 없이 노트북 셀 단위로 순차 실행하더라도 각 미션이 해당 모듈의 역할을 1:1로 수행하며, **미션 10(시스템 통합)**에서 전체 파이프라인이 유기적으로 완성되도록 설계되어 있습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 02. 게시물 텍스트 불용어 정제, 이미지 해시(pHash) 추출 및 중복 제거 전처리 모듈 구현
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

텍스트 불용어 제거, 특수문자 정제, 이미지 지각 해시(pHash) 추출을 통해 중복 게시물을 걸러내는 전처리 모듈을 구현한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **게시물 텍스트의 불용어·특수문자·URL·이모지를 정제하고 이미지 지각 해시(pHash)를 추출해 중복 게시물을 걸러내는 전처리 모듈을 구현한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **유니코드 정규화(전각 문자 등), URL·멘션·해시태그 기호·특수문자·이모지·불용어 제거, 본문 결측 보정을 구현하고, 32x32 이미지의 DCT 저주파 성분으로 64비트 pHash 작성하여 원인을 분석하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 5개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/preprocess/isr02_preprocessor.py 및 전처리 전후 데이터셋` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 전술 환경 특유의 노이즈(스푸핑, 반사파, 센서 단절 등) 제거 타당성

#### 미션 PASS 조건

- 정량 기준: 데이터 정제 처리 속도 1,000건/초 이상, 전처리 오류율 0%


- 진행 기준: 정제 파이프라인 자동화 및 단위 테스트 케이스 5종 통과


- 검증 방법: pytest 기반 전처리 정합성 테스트 및 결측치 보정율 확인


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""SNS 게시물 텍스트의 유니코드 정규화·URL·멘션·특수문자·이모지·불용어 제거와 결측 처리, 이미지 지각 해시(pHash) 추출을 수행하고, 정규화 본문과 이미지 해시로 중복 게시물을 걸러내는 전처리 모듈을 구현해 처리 속도와 정합성을 검증한다."""
import re
import time
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage
from scipy.fft import dctn

# ---------------------------------------------------------------------------
# [교육용가상자료] 합성 SNS 게시물 생성기 (실제 계정·채널·지명·게시물 없음)
# 가상 사건마다 원본 제보, 재게시(복제), 유사 재작성, 조작·기만 게시물 10종과 32x32 합성 이미지를 만든다.
# ---------------------------------------------------------------------------
REGIONS = [f"{a}-{n}" for a in "ABCDEF" for n in range(1, 6)]
OBJECTS = ["군용 차량", "소형 드론", "보급 트럭", "헬기", "버스", "견인 차량", "트레일러", "민간 승용차"]
DIRS = ["북쪽", "남쪽", "동쪽", "서쪽"]
TEMPLATES = ["가상 {r} 지역에서 {o} 목격, {n}대 규모로 {d}으로 이동 중",
             "{r} 일대 {o} 확인됨. 약 {n}대, {d} 방향으로 이동",
             "[현지] {r}에서 {o} 발견 - 수량 {n}, 이동 방향 {d}",
             "{r} 주민 제보: {o} {n}대가 {d}으로 향하는 모습"]
MANIP = ["num_change", "region_swap", "dir_flip", "old_image", "splice", "ai_image", "ai_text", "bot_burst", "mashup", "recycled"]
ACCOUNT_TYPES = {  # 유형별 계정 속성 분포(가상): 나이(일), 팔로워, 하루 게시 수, 원본 비율, 과거 정확도, 게시 간격 변동계수
    "media": ((1000, 4000), (5e4, 5e5), (5, 30), (.6, .9), (.85, .97), (.8, 1.5)),
    "witness": ((10, 800), (20, 2000), (.2, 3), (.7, 1.0), (.6, .95), (1.0, 2.0)),
    "aggregator": ((200, 2000), (1e4, 2e5), (30, 200), (.05, .2), (.6, .85), (.5, 1.0)),
    "propaganda": ((5, 200), (100, 2e4), (20, 150), (.3, .7), (.1, .4), (.05, .3)),
}


def make_accounts(rng, n=(8, 30, 10, 20)):
    acc = {}
    for (t, spec), k in zip(ACCOUNT_TYPES.items(), n):
        for i in range(k):
            v = [rng.uniform(*r) for r in spec]
            acc[f"ACC_{t[:3].upper()}{i:02d}"] = {"type": t, "age_days": int(v[0]), "followers": int(v[1]), "posts_per_day": round(v[2], 1),
                                                   "original_ratio": round(v[3], 2), "past_accuracy": round(v[4], 2), "interval_cv": round(v[5], 2)}
    return acc


def event_image(seed):
    """사건 현장 사진을 흉내 낸 32x32 합성 이미지(사건마다 고유한 도형 배치)."""
    # [TODO : 학생 실습 2] 게시물 텍스트 불용어 정제, 이미지 해시(pHash) 추출 및 중복 제거 전처리 모듈 구현 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def variant(img, rng):
    """재게시 과정의 자연스러운 변형: 재압축 잡음, 밝기 변화, 가장자리 잘림 중 하나."""
    # [TODO : 학생 실습 2] 게시물 텍스트 불용어 정제, 이미지 해시(pHash) 추출 및 중복 제거 전처리 모듈 구현 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def splice(img, other, rng):
    out = img.copy()
    y, x = rng.integers(4, 18, 2)
    out[y:y + 12, x:x + 12] = other[y:y + 12, x:x + 12] + rng.normal(0, 12, (12, 12))   # 잡음 수준이 다른 조각을 붙임
    return out


def ai_image(rng):
    """생성형 모델 이미지를 흉내 낸 합성 이미지: 지나치게 매끈한 바탕 + 업샘플링 격자 무늬(주기 2픽셀)."""
    base = ndimage.gaussian_filter(rng.normal(130, 60, (32, 32)), 3)
    grid = (np.indices((32, 32)).sum(0) % 2) * rng.uniform(6, 12)
    return base + grid


def write(ev, rng, tpl=None, **kw):
    s = {"r": ev["r"], "o": ev["o"], "n": ev["n"], "d": ev["d"], **kw}
    return (tpl or TEMPLATES[int(rng.integers(len(TEMPLATES)))]).format(**s)


def make_posts(n_events=40, seed=7):
    rng = np.random.default_rng(seed)
    acc = make_accounts(rng)
    by = {t: [a for a, v in acc.items() if v["type"] == t] for t in ACCOUNT_TYPES}
    events = [{"id": f"EV{i:03d}", "r": str(rng.choice(REGIONS)), "o": str(rng.choice(OBJECTS)), "n": int(rng.integers(2, 15)),
               "d": str(rng.choice(DIRS)), "t0": float(i * 1.2 + rng.uniform(0, 1)), "img": event_image(seed * 1000 + i)} for i in range(n_events)]
    posts = []

    def add(ev, account, text, img, t, fwd, kind, origin):
        pid = f"P{len(posts):04d}"
        posts.append({"post_id": pid, "platform": str(rng.choice(["telegram", "x", "other"], p=[.6, .3, .1])), "account_id": account,
                      "posted_at": round(t, 3), "text": text, "image": img, "forwarded_from": fwd, "event_id": ev["id"] if ev else None,
                      "kind": kind, "is_disinfo": kind in MANIP, "origin": origin})
        return pid

    for k, ev in enumerate(events):
        src = str(rng.choice(by["witness"] if rng.random() < .6 else by["media"]))
        if rng.random() < .05:
            src = str(rng.choice(by["propaganda"]))                      # 신뢰도 낮은 계정도 사실을 올릴 수 있다
        o = add(ev, src, write(ev, rng), ev["img"], ev["t0"], None, "original", None)
        for _ in range(int(rng.integers(2, 5))):                          # 그대로 재게시(전달)
            add(ev, str(rng.choice(by["aggregator"])), posts[int(o[1:])]["text"], ev["img"], ev["t0"] + rng.exponential(1), o, "copy", o)
        for _ in range(int(rng.integers(1, 3))):                          # 표현만 바꾼 재작성(내용 동일)
            add(ev, str(rng.choice(by["media"] + by["witness"])), "#가상속보 " + write(ev, rng), variant(ev["img"], rng),
                ev["t0"] + rng.uniform(0.5, 6), None, "near_dup", o)
        for m in MANIP if k % 2 == 0 else MANIP[k % 4::4]:                # 조작·기만 게시물
            if rng.random() > 0.5:
                continue
            who = str(rng.choice(by["propaganda"] if rng.random() < .75 else by["aggregator"] + by["witness"]))
            t, img, text = ev["t0"] + rng.uniform(1, 12), variant(ev["img"], rng), None
            if m == "num_change":
                text = write(ev, rng, n=ev["n"] * int(rng.integers(5, 10)))
            elif m == "region_swap":
                text = write(ev, rng, r=str(rng.choice([r for r in REGIONS if r != ev["r"]])))
            elif m == "dir_flip":
                text = write(ev, rng, d={"북쪽": "남쪽", "남쪽": "북쪽", "동쪽": "서쪽", "서쪽": "동쪽"}[ev["d"]])
            elif m == "old_image" and k > 0:
                text, img = write(ev, rng), events[int(rng.integers(0, k))]["img"]
            elif m == "splice":
                text, img = write(ev, rng), splice(ev["img"], events[(k + 7) % n_events]["img"], rng)
            elif m == "ai_image":
                text, img = write(ev, rng), ai_image(rng)
            elif m == "ai_text":
                text = f"속보 속보! {ev['r']} 지역 대규모 {ev['o']} 출현! 긴급 확산 바람! 긴급 확산 바람! 모두 확산 바람!"
                img = ai_image(rng) if rng.random() < .5 else None
            elif m == "bot_burst":
                text = write(ev, rng, n=ev["n"] * 6) + " 지금 바로 공유"
                for b in rng.choice(by["propaganda"], 4, replace=False):
                    add(ev, str(b), text, img, t + rng.uniform(0, 0.15), None, m, o)
                continue
            elif m == "mashup":
                other = events[(k + 3) % n_events]
                text = f"{ev['r']} 일대 {ev['o']} 확인, {other['n']}대가 {other['d']}으로 이동"   # 다른 사건 내용과 짜깁기
            elif m == "recycled" and k > 2:
                old = events[int(rng.integers(0, k - 1))]
                text, img, t = write(old, rng), old["img"], ev["t0"] + rng.uniform(0, 3)
            if text is None:
                continue
            add(ev, who, text, img, t, None, m, o)
    return posts, acc
# ---------------------------------------------------------------------------

# ---- 텍스트 정제·이미지 지각 해시·조작 흔적 공통 함수 ----
STOPWORDS = {"가상", "약", "일대", "지역에서", "모습", "중", "확인됨", "발견", "제보", "주민"}
_URL = re.compile(r"https?://\S+|www\.\S+")
_SPECIAL = re.compile(r"[^0-9A-Za-z가-힣\-\s]")


def clean_text(t):
    """정규화(NFKC) -> URL·멘션 제거 -> 해시태그 기호 제거 -> 특수문자 제거 -> 불용어 제거 -> 공백 정리."""
    t = unicodedata.normalize("NFKC", t)
    t = _URL.sub(" ", t)
    t = re.sub(r"@\w+", " ", t).replace("#", " ")
    t = _SPECIAL.sub(" ", t)
    return " ".join(w for w in t.split() if w not in STOPWORDS)


def phash(img):
    """지각 해시(pHash): 32x32 이미지 DCT의 저주파 8x8(직류 성분 제외)을 중앙값과 비교한 64비트."""
    c = dctn(np.asarray(img, float), norm="ortho")[:8, :8].ravel()[1:]
    return np.append(c > np.median(c), False)


def hamming(a, b):
    return int(np.count_nonzero(a != b))


def slots(text):
    """본문에서 지역 코드·수량·방향·대상을 뽑는다(사건 내용 비교용)."""
    r = re.search(r"[A-F]-\d", text)
    n = re.search(r"(\d+)\s*대|수량\s*(\d+)", text)
    d = next((x for x in DIRS if x in text), None)
    o = next((x for x in OBJECTS if x in text), None)
    return {"r": r.group(0) if r else None, "n": int(n.group(1) or n.group(2)) if n else None, "d": d, "o": o}


def splice_score(img):
    """합성(붙여넣기) 흔적: 8x8 블록별 고주파 잡음 크기의 최대/중앙값 비율. 잡음 수준이 다른 조각이 있으면 커진다."""
    res = img - ndimage.median_filter(img, 3)
    b = np.abs(res).reshape(4, 8, 4, 8).std(axis=(1, 3))
    return float(b.max() / (np.median(b) + 1e-6))


def ai_image_score(img):
    """생성형 이미지 흔적: 2픽셀 주기 격자(나이퀴스트 주파수) 에너지 비율."""
    f = np.abs(np.fft.fft2(img - img.mean()))
    return float(f[16, 16] / (f.sum() / f.size + 1e-6))


def ai_text_score(text):
    """기계적 선동 문체 흔적: 반복 어절 비율 + 느낌표 밀도."""
    w = text.split()
    return (1 - len(set(w)) / max(len(w), 1)) + text.count("!") / max(len(w), 1)

posts, acc = make_posts(60, seed=7)
rng = np.random.default_rng(3)
FIRE, FLAG = chr(0x1F525), chr(0x1F6A8)                  # 이모지(코드값으로 생성)
for p in posts:                                          # 수집 과정의 잡음 주입
    r = rng.random()
    if r < .15:
        p["text"] += f" https://example.invalid/{rng.integers(1e6)}"
    elif r < .3:
        p["text"] = "@ch_virtual " + p["text"] + " " + FIRE + FLAG
    elif r < .4:
        p["text"] = unicodedata.normalize("NFKC", p["text"]).replace("A", "Ａ").replace("1", "１")   # 전각 문자
    elif r < .43:
        p["text"] = None                                 # 본문 결측(이미지만 게시)

# 1) 처리 속도와 오류율
t0, out, fails = time.perf_counter(), [], 0
for p in posts:
    try:
        txt = clean_text(p["text"] or "")
        out.append({"id": p["post_id"], "t": p["posted_at"], "clean": txt, "missing": p["text"] is None,
                    "hash": phash(p["image"]) if p["image"] is not None else None})
    except Exception:
        fails += 1
dt = time.perf_counter() - t0
print(f"전처리 {len(posts)}건, {len(posts) / dt:,.0f}건/초 (목표 1,000건/초 이상), 실패 {fails}건 -> 오류율 {fails / len(posts):.0%}")
miss = [o for o in out if o["missing"]]
print(f"본문 결측 {len(miss)}건: 빈 문자열로 보정하고 '이미지 단독 게시물'로 표시(보정율 100%)")

# 2) 중복 제거: 같은 정규화 본문 + 이미지 해시 거리 4 이하(또는 둘 다 없음)는 같은 게시물로 보고 가장 이른 것만 남긴다
kept, dup_of = [], {}
for o in sorted(out, key=lambda x: x["t"]):
    same = next((k for k in kept if k["clean"] == o["clean"] and o["clean"] and
                 ((k["hash"] is None and o["hash"] is None) or (k["hash"] is not None and o["hash"] is not None and hamming(k["hash"], o["hash"]) <= 4))), None)
    if same:
        dup_of[o["id"]] = same["id"]
    else:
        kept.append(o)
kinds = {p["post_id"]: p["kind"] for p in posts}
print(f"중복 제거: {len(out)}건 -> {len(kept)}건 (제거 {len(dup_of)}건, 그중 실제 재게시(copy) {sum(kinds[d] == 'copy' for d in dup_of)}건 / 조직적 동시 게시 {sum(kinds[d] == 'bot_burst' for d in dup_of)}건)")
print(f"전각 문자·이모지·URL 차이는 정제로 흡수되고, 본문 결측 등으로 남은 재게시 {sum(kinds[k['id']] == 'copy' for k in kept)}건은 미션 3의 유사도 판별로 넘긴다")

# 3) 단위 테스트 5종
e1, e2 = event_image(1), event_image(2)
tests = {
    "URL·멘션 제거": clean_text("@abc 보고 https://x.invalid/1 확인") == "보고 확인",
    "이모지·특수문자 제거": FIRE not in clean_text("정전 " + FIRE + "!!") and "!" not in clean_text("정전!!"),
    "전각 문자 정규화(NFKC)": clean_text("Ａ-１ 정전") == "A-1 정전",
    "pHash 재압축 강건성(거리 8 이하)": hamming(phash(e1), phash(ndimage.gaussian_filter(e1, 0.6) + rng.normal(0, 3, e1.shape))) <= 8,
    "pHash 다른 사진 구분(거리 16 이상)": hamming(phash(e1), phash(e2)) >= 16,
}
for k, v in tests.items():
    print(f"[{'PASS' if v else 'FAIL'}] {k}")
print(f"단위 테스트 {sum(tests.values())}/5 통과")

# 4) 정제 전후 예시
ex = [next(p for p in posts if p["text"] and key in p["text"]) for key in ("http", "@", "\uff21")]
print("\n[정제 전후 예시]")
for p in ex:
    print(" 전:", p["text"].replace(FIRE, "<U+1F525>").replace(FLAG, "<U+1F6A8>"))
    print(" 후:", clean_text(p["text"]))

d_same = [hamming(phash(p["image"]), phash(posts[int(p["origin"][1:])]["image"])) for p in posts if p["kind"] in ("copy", "near_dup") and p["image"] is not None]
d_diff = [hamming(phash(posts[i]["image"]), phash(posts[j]["image"])) for i, j in rng.integers(0, len(posts), (200, 2))
          if posts[i]["event_id"] != posts[j]["event_id"] and posts[i]["image"] is not None and posts[j]["image"] is not None]
plt.figure(figsize=(7, 4))
plt.hist([d_same, d_diff], bins=32, range=(0, 64), label=["same event (repost)", "different events"])
plt.axvline(8, ls="--", c="k"); plt.xlabel("pHash Hamming distance"); plt.ylabel("pairs"); plt.legend(); plt.title("Perceptual hash distance (synthetic images)")
plt.show()


<details>
<summary><b>💡 [참고] 미션 02 예상 실행 결과 (Output)</b></summary>

```text
전처리 573건, 6,551건/초 (목표 1,000건/초 이상), 실패 0건 -> 오류율 0%
본문 결측 14건: 빈 문자열로 보정하고 '이미지 단독 게시물'로 표시(보정율 100%)
중복 제거: 573건 -> 340건 (제거 233건, 그중 실제 재게시(copy) 169건 / 조직적 동시 게시 53건)
전각 문자·이모지·URL 차이는 정제로 흡수되고, 본문 결측 등으로 남은 재게시 6건은 미션 3의 유사도 판별로 넘긴다
[PASS] URL·멘션 제거
[PASS] 이모지·특수문자 제거
[PASS] 전각 문자 정규화(NFKC)
[PASS] pHash 재압축 강건성(거리 8 이하)
[PASS] pHash 다른 사진 구분(거리 16 이상)
단위 테스트 5/5 통과

[정제 전후 예시]
 전: [현지] A-4에서 보급 트럭 발견 - 수량 9, 이동 방향 서쪽 https://example.invalid/179440
 후: 현지 A-4에서 보급 트럭 - 수량 9 이동 방향 서쪽
 전: @ch_virtual [현지] A-4에서 보급 트럭 발견 - 수량 9, 이동 방향 동쪽 <U+1F525><U+1F6A8>
 후: 현지 A-4에서 보급 트럭 - 수량 9 이동 방향 동쪽
 전: Ａ-2 주민 제보: 버스 8대가 서쪽으로 향하는 모습
 후: A-2 버스 8대가 서쪽으로 향하는
```
</details>
